In [ ]:
import numpy as np
import pandas as pd
import glob
import os
import subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import glob
from collections import defaultdict
conda_env_name="bioinfo"

In [ ]:
# nazwy folderów

main_dir=Path(f"/mnt/storage_10T/Sano_internship/MariaWiluszData")

results_dir=Path(f"{main_dir}/all_duf_results/deepfri")
os.makedirs(results_dir, exist_ok=True)

known_dir=Path(f"{results_dir}/known")# potem w tych folderach będą osobno chainsaw i pfam w osobnych podfolderach 
# a w nich jeszcze będzie full structure i statystyki go termów, więc po 4 foldery z wynikami na domene
os.makedirs(known_dir, exist_ok=True)

unknown_dir=Path(f"{results_dir}/unknown")
os.makedirs(unknown_dir, exist_ok=True)

data_dir=Path(f"{main_dir}/data")


ic_file_path="/mnt/storage_10T/Sano_internship/data/GO_information_content_swiss-model.txt"


# deepfri predict.py location
deepfri_script=Path("/mnt/storage_10T/Sano_internship/software/deepfri_onnx")


# Definicje funkcji

In [ ]:
def domains_to_dict(file_name):
    '''Funkcja zamieniająca csv z kolumną domain_number i accessions białek po tej kolumnie
    w słownik z domain_number jako kluczem i kolejnymi accessions białek w liście jako wartości'''
    
    df=pd.read_csv(file_name)
    domain_number_index=df.columns.get_loc("domain_number")
    df=df.iloc[:,[domain_number_index,domain_number_index+1]]
    
    return df.groupby("domain_number")[df.columns[1]].apply(list).to_dict()
    

In [ ]:
def domains_to_list(file_name):
    '''Funkcja zamieniająca csv z kolumną domain_number w listę numerów domen'''
    
    return list(set(pd.read_csv(file_name)["domain_number"]))

In [ ]:
# trzeba do tej funkcji używać jednak folderów a nie plików jak wcześniej więc później jak coś można dodać 
# usuwanie dodatkowych accessions z wyników i do tego używać tego słownika

def DeepFRI_on_list(domain_list,known,chainsaw):# base_results_dir to (un)known_dir
    '''Funkcja puszczająca DeepFRI na przygotowanej przez domains_to_list() liście.
    Nic nie zwraca, zapisuje wyniki w csv w dwóch docelowych folderach'''

    for domain in domain_list:
        #accessions=domain_dict[domain]
        
        if known==True:
            # directory for deepfri output
            if chainsaw==True:
                deepfri_dir=Path(f"{known_dir}/{domain}_deepfri_chainsaw")
            elif chainsaw==False:
                deepfri_dir=Path(f"{known_dir}/{domain}_deepfri_pfam")
        elif known==False:
            if chainsaw==True:
                deepfri_dir=Path(f"{unknown_dir}/{domain}_deepfri_chainsaw")
            elif chainsaw==False:
                deepfri_dir=Path(f"{unknown_dir}/{domain}_deepfri_pfam")
        os.makedirs(deepfri_dir, exist_ok=True)
        
        os.chdir(deepfri_script)
        print("New working directory:", Path.cwd())
    
        output_prefix=deepfri_dir / f"{domain}_all_fragments"
    
        if chainsaw==True:
            accessions_dir=Path(f"{data_dir}/{domain}_chainsaw_structures")
        elif chainsaw==False:
            accessions_dir=Path(f"{data_dir}/{domain}_extracted_structures")
        deepfri_command=[
            "conda",
            "run",
            "-n",
            conda_env_name,
            "python",
            f"{str(deepfri_script)}/predict.py",
            "--pdb_dir",
            str(accessions_dir),
            #str(accession_file),
            "-ont",
            "mf", "bp", "cc",
            "--output_fn_prefix",
            str(output_prefix)
        ]
        process=subprocess.run(
            deepfri_command,
            capture_output=True,
            text=True,
        )

        if process.returncode == 0:
            print(f"DeepFRI processing complete.")
            print(f"DeepFRI output:\n{process.stdout}")
        else:
            print(f"Error running DeepFRI. Return code: {process.returncode}")
            print(f"DeepFRI stdout:\n{process.stdout}")
            print(f"DeepFRI stderr:\n{process.stderr}")
    
        

        # Go back
        os.chdir("/mnt/storage_10T/Sano_internship")

# Sam DeepFri na domenach z Pfam i Chainsaw


Wyniki liczyły się przez około 7 godzin, dla wszystkich 4 kategorii

## Na domenach o znanej funkcji

In [ ]:
# domeny z Pfam
DeepFRI_on_list(list(domains_to_dict(f"{main_dir}/accessions_random_100_known_function.csv").keys()),known=True,chainsaw=False)

#to pewnie też działa
#DeepFRI_on_list(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False)


In [ ]:
# domeny z Chainsaw
DeepFRI_on_list(list(domains_to_dict(f"{main_dir}/all_duf_results/lengths/known_100_lengths.csv").keys()),known=True,chainsaw=True)


## Na domenach o nieznanej funkcji

In [ ]:
# domeny z Pfam
DeepFRI_on_list(list(domains_to_dict(f"{main_dir}/accessions_random_100_unknown_function.csv").keys()),known=False,chainsaw=False)


In [ ]:
# domeny z Chainsaw
DeepFRI_on_list(list(domains_to_dict(f"{main_dir}/all_duf_results/lengths/unknown_100_lengths.csv").keys()),known=False,chainsaw=True)
